### Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay

from xgboost import XGBClassifier
import joblib

### Load Cleaned Dataset

In [ ]:
df = pd.read_csv('/content/cleaned_incidents.csv')

print("Dataset Loaded ")
print("Shape:", df.shape)

### Split Features & Target

In [ ]:
X = df.drop('closed_code', axis=1)
y = df['closed_code']

### Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

### Initialize Models

In [ ]:
models = {
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "SVM": SVC(),
    "XGBoost": XGBClassifier(eval_metric='mlogloss', use_label_encoder=False)
}

### Train & Evaluate Models

In [ ]:
results = {}

for name, model in models.items():
    print(f"\nTraining {name}...")

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='macro')

    results[name] = {
        "model": model,
        "accuracy": acc,
        "f1": f1,
        "y_pred": y_pred
    }

    print(f"{name} Accuracy: {acc:.4f}")
    print(f"{name} F1 Score: {f1:.4f}")

### Compare Models (Table)

In [ ]:
results_df = pd.DataFrame({
    name: [res["accuracy"], res["f1"]]
    for name, res in results.items()
}, index=["Accuracy", "F1 Score"]).T

print("\nModel Comparison:\n")
print(results_df)

### Select Best Model (Based on F1)

In [ ]:
best_model_name = max(results, key=lambda x: results[x]["f1"])
best_model = results[best_model_name]["model"]

print(f"\nBest Model: {best_model_name}")

### Save Best Model

In [ ]:
joblib.dump(best_model, '/content/best_model.pkl')

print("Model saved successfully ✅")

### Bar Chart Comparison

In [ ]:
names = list(results.keys())
accuracy_vals = [results[name]["accuracy"] for name in names]
f1_vals = [results[name]["f1"] for name in names]

x = np.arange(len(names))

plt.figure()
plt.bar(x - 0.2, accuracy_vals, width=0.4)
plt.bar(x + 0.2, f1_vals, width=0.4)

plt.xticks(x, names)
plt.xlabel("Models")
plt.ylabel("Score")
plt.title("Model Comparison (Accuracy vs F1 Score)")
plt.legend(["Accuracy", "F1 Score"])

plt.show()

### Confusion Matrix (Best Model)

In [ ]:
y_pred_best = results[best_model_name]["y_pred"]

cm = confusion_matrix(y_test, y_pred_best)

disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot()

plt.title(f"Confusion Matrix - {best_model_name}")
plt.show()